In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random

pi = 3.14159265359

maxval=1e9
minval=1e-9

#import OptimizedDataGenerator as DG
#from loss import *
#from loss_kl import *
#from loss_hybrid2 import * 


2025-09-22 03:19:57.275605: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-22 03:19:57.275692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-22 03:19:57.396076: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-22 03:19:57.881721: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-22 03:20:25.113920: W tensorflow/compiler/tf2

In [2]:
#from loss_functions.loss_nll import *
from loss_functions.Slim_SSE import *
#from models.models import *
from models.Slim1D_model_Compression import *
import utils

2025-09-22 03:21:34.144302: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3234 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 1g.5gb, pci bus id: 0000:81:00.0, compute capability: 8.0


In [3]:
# try auto sf
#import OptimizedDataGenerator as DG
#import OptimizedDataGenerator as DG
import OptimizedDataGenerator_v2 as DG


In [4]:
batch_size = 5000
val_batch_size = 5000
train_file_size = 68
val_file_size = 12

Training

In [5]:
def count_files_in_directory(directory_path):
    try:
        return len([f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))])
    except FileNotFoundError:
        print(f"Directory not found: {directory_path}")
        return 0


In [6]:
data_directory_path = '/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/contained/train'

In [7]:
tfrecords_dir_train = "/depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_train_run3"
tfrecords_dir_validation = "/depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_val_run3"


In [8]:
# Count files in both directories
data_files_count = count_files_in_directory(data_directory_path)


# Print results
print(f"Number of files in data directory: {data_files_count}")


Number of files in data directory: 80


Create a new tfrecord can avoid the problem of not converging. 

In [9]:
#utils.safe_remove_directory(tfrecords_dir_train)
#utils.safe_remove_directory(tfrecords_dir_validation)

In [11]:
validation_generator = DG.OptimizedDataGenerator(
    # Base directory where your parquet files are located
    dataset_base_dir = data_directory_path,
    is_directory_recursive = False,
    batch_size = val_batch_size,
    file_count = val_file_size,
    to_standardize = True,
    include_y_local = False,
    # Labels you want to extract
    labels_list = ['x-midplane', 'y-midplane', 'cotBeta'],  
    # Input shape for the model (time slices, height, width)
    input_shape = (20, 16, 16),
    transpose = (0, 2, 3, 1),
    shuffle = False,
    files_from_end = False,
    
    # New parameters specific to this generator
    tfrecords_dir = tfrecords_dir_validation,
    use_time_stamps = -1, #[0, 19],  # Specific time stamps to use
    quantize = False,  # Whether to use quantization during loading
    max_workers = 2  # For parallel processing
)

Processing Files...: 100%|██████████| 12/12 [00:11<00:00,  1.09it/s]


Directory /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_val_run3 is removed...


Saving batches as TFRecords: 100%|██████████| 14/14 [00:25<00:00,  1.85s/it]


Metadata saved successfully ast /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_val_run3/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_val_run3/metadata.json


In [12]:
train_generator = DG.OptimizedDataGenerator(
    # Base directory where your parquet files are located
    dataset_base_dir = data_directory_path,
    is_directory_recursive = False,
    batch_size = batch_size,
    file_count = train_file_size,
    to_standardize = True,
    include_y_local = False,
    # Labels you want to extract
    labels_list = ['x-midplane', 'y-midplane', 'cotBeta'],  
    # Input shape for the model (time slices, height, width)
    input_shape = (20, 16,16),
    transpose = (0, 2, 3, 1),
    shuffle = False,
    files_from_end = True,  # Using files from the end of the list
    
    # New parameters specific to this generator
    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = -1, #[0, 19],  # Specific time stamps to use
    quantize = False,  # Whether to use quantization during loading
    max_workers = 2  # For parallel processing
)

Processing Files...: 100%|██████████| 68/68 [01:02<00:00,  1.08it/s]


Directory /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_train_run3 is removed...


Saving batches as TFRecords: 100%|██████████| 74/74 [02:02<00:00,  1.65s/it]


Metadata saved successfully ast /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_train_run3/metadata.json
Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/tf_records/baseline_16x16_3sr_train_run3/metadata.json


In [13]:
train_generator.labels_scale

array([99.18727961, 24.80349315,  1.91469723])

In [14]:
input_shape = (16, 16, 20)
model = CreateSlim1DModel(
    input_shape=input_shape,
    
    # ========== MODEL SIZE PARAMETERS ==========
    conv_filters=5,           # Conv1D filters (5 original)
    conv_kernel_size=3,       # Conv1D kernel size (3 original)
    dense1_units=16,          # First dense layer size (16 original)
    dense2_units=16,          # Second dense layer size (16 original)
    output_units=3,           # Output size (keep at 3 for x,y,β)
    
    # ========== QUANTIZATION PARAMETERS ==========
    use_quantization=False,   # True/False to enable QKeras
    conv_weight_bits=4,       # Conv1D weight precision (4 like baseline)
    conv_activation_bits=4,   # Conv1D activation precision (4 like baseline)
    dense_weight_bits=8,      # Dense weight precision (8 like baseline)
    dense_activation_bits=8,  # Dense activation precision (8 like baseline)
    
    # ========== PRUNING PARAMETERS ==========
    use_pruning=False,        # True/False to enable pruning
    pruning_fraction=0.3,     # 0.2=20%, 0.5=50%, 0.8=80% weights removed
    pruning_schedule='constant'  # 'constant' or 'polynomial_decay'
)
model.summary()

Model: "smrtpxl_slim_tunable"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls/ (InputLayer)    [(None, 16, 16, 20)]         0         []                            
                                                                                                  
 avg_pooling_2d_proj_x (Ave  (None, 16, 1, 20)            0         ['input_pxls/[0][0]']         
 ragePooling2D)                                                                                   
                                                                                                  
 avg_pooling_2d_proj_y (Ave  (None, 1, 16, 20)            0         ['input_pxls/[0][0]']         
 ragePooling2D)                                                                                   
                                                                               

In [15]:
'''
decay_steps = 90*train_generator.__len__()
alpha = 0.01
initial_learning_rate = 1e-3
warmup_target = 1e-1
warmup_steps = 10*train_generator.__len__()
'''
decay_steps = 90*train_generator.__len__()
alpha = 0.01
initial_learning_rate = 1e-5
warmup_target = 1e-3
warmup_steps = 10*train_generator.__len__()

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    alpha=alpha,
    warmup_target = warmup_target,
    warmup_steps = warmup_steps
)



In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=slim_sse_loss,  # Use slim SSE loss
    metrics=[slim_rmse_metric, x_mae_metric, y_mae_metric, cotBeta_mae_metric]  # Optional: add metrics
)

In [17]:
fingerprint = '%08x' % random.randrange(16**8)
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [18]:
checkpoint_filepath

'./trained_models/model-93677f46-checkpoints/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [19]:
print(fingerprint)

93677f46


In [20]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback

early_stopping_patience = 50

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort()
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [21]:
class NanMonitorCallback(tf.keras.callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        if loss is not None:
            if np.isnan(loss) or np.isinf(loss):
                print('NaN loss detected, terminating training')
                self.model.stop_training = True
                
# Add this to your callbacks list
nan_monitor = NanMonitorCallback()
callbacks = [es, mcp, csv_logger, nan_monitor]

In [ ]:
# Now use it in model.fit()
history = model.fit(
    x=train_generator,
    validation_data=validation_generator,
    callbacks=callbacks,
    epochs=1000,
    shuffle=False,
    verbose=1
)

Epoch 1/1000


2025-09-22 03:25:40.823394: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-09-22 03:25:40.944452: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2025-09-22 03:25:41.229023: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-09-22 03:25:42.447093: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f7a0951dbe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-22 03:25:42.447151: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 1g.5gb, Compute Capability 8.0
2025-09-22 03:25:42.465890: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1758504342.593817  754174 device_

74/74 [==============================] - ETA: 0s - loss: 2068.8665 - slim_rmse_metric: 0.3488 - x_mae_metric: 0.2983 - y_mae_metric: 0.3396 - cotBeta_mae_metric: 0.2078
Epoch 1: val_loss improved from inf to 525.93799, saving model to ./trained_models/model-93677f46-checkpoints/weights.01-t2068.87-v525.94.hdf5
74/74 [==============================] - 21s 215ms/step - loss: 2068.8665 - slim_rmse_metric: 0.3488 - x_mae_metric: 0.2983 - y_mae_metric: 0.3396 - cotBeta_mae_metric: 0.2078 - val_loss: 525.9380 - val_slim_rmse_metric: 0.1906 - val_x_mae_metric: 0.1433 - val_y_mae_metric: 0.1800 - val_cotBeta_mae_metric: 0.0982
Epoch 2/1000
74/74 [==============================] - ETA: 0s - loss: 432.3430 - slim_rmse_metric: 0.1688 - x_mae_metric: 0.1300 - y_mae_metric: 0.1546 - cotBeta_mae_metric: 0.0808
Epoch 2: val_loss improved from 525.93799 to 343.47720, saving model to ./trained_models/model-93677f46-checkpoints/weights.02-t432.34-v343.48.hdf5
74/74 [==============================] - 24s

In [24]:
fingerprint

'93677f46'

In [ ]:
1

In [ ]:
1